In [ ]:

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

print("\n================ DEVICE INFORMATION ================")

if torch.cuda.is_available():

    device = torch.device("cuda")
    print("Using GPU")
    print("GPU:", torch.cuda.get_device_name(0))

else:

    device = torch.device("cpu")
    print("Using CPU")
train_file = "train.csv"
test_file = "test.csv"

train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)

print("\n================ DATASET INFORMATION ================")

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

print("\nTrain Columns:")
print(train_df.columns.tolist())

print("\nTest Columns:")
print(test_df.columns.tolist())

print("\nFirst 5 Training Records:")
display(train_df.head())
train_df.columns = train_df.columns.str.strip()
test_df.columns = test_df.columns.str.strip()

train_df = train_df.rename(
    columns={
        "Class Index": "label",
        "Title": "title",
        "Description": "description"
    }
)

test_df = test_df.rename(
    columns={
        "Class Index": "label",
        "Title": "title",
        "Description": "description"
    }
)

print("\nColumns after renaming:")
print(train_df.columns.tolist())
print("\n================ MISSING VALUES ================")

print("\nTrain Missing Values:")
print(train_df.isnull().sum())

print("\nTest Missing Values:")
print(test_df.isnull().sum())

train_df["title"] = train_df["title"].fillna("")
train_df["description"] = train_df["description"].fillna("")

test_df["title"] = test_df["title"].fillna("")
test_df["description"] = test_df["description"].fillna("")

train_df["label"] = pd.to_numeric(
    train_df["label"],
    errors="coerce"
)

test_df["label"] = pd.to_numeric(
    test_df["label"],
    errors="coerce"
)

train_df = train_df.dropna(subset=["label"])
test_df = test_df.dropna(subset=["label"])

train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

print("\n================ LABEL DISTRIBUTION ================")

print(
    train_df["label"].value_counts().sort_index()
)

train_df["label"] = train_df["label"] - 1
test_df["label"] = test_df["label"] - 1

train_df["text"] = (
    train_df["title"].astype(str)
    + " "
    + train_df["description"].astype(str)
)

test_df["text"] = (
    test_df["title"].astype(str)
    + " "
    + test_df["description"].astype(str)
)

print("\n================ REDUCING DATASET ================")

train_df = train_df.sample(
    n=min(8000, len(train_df)),
    random_state=42
).reset_index(drop=True)

test_df = test_df.sample(
    n=min(2000, len(test_df)),
    random_state=42
).reset_index(drop=True)

print("Reduced Training Shape:", train_df.shape)
print("Reduced Testing Shape:", test_df.shape)
train_df = train_df[["text", "label"]].copy()
test_df = test_df[["text", "label"]].copy()

print("\n================ CLEAN DATA ================")

display(train_df.head())

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

print("\n================ HUGGING FACE DATASET ================")

print("Training Dataset:", train_dataset)
print("Testing Dataset:", test_dataset)

model_name = "distilbert-base-uncased"

print("\n================ TOKENIZER ================")

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("\nTokenizing training data...")

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

print("Training tokenization completed!")

print("\nTokenizing testing data...")

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print("Testing tokenization completed!")

tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

print("Tokenized datasets ready!")
id2label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

label2id = {
    "World": 0,
    "Sports": 1,
    "Business": 2,
    "Sci/Tech": 3
}

print("\n================ MODEL ================")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

model.to(device)

print("DistilBERT model loaded successfully!")
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }
training_args = TrainingArguments(
    output_dir="./news_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    report_to="none"

)
print("\nTraining arguments created successfully!")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")
print("\n")
print("============================================================")
print("STARTING DISTILBERT TRAINING")
print("============================================================")
trainer.train()
print("\nTraining completed successfully!")
print("\n")
print("============================================================")
print("MODEL EVALUATION")
print("============================================================")
results = trainer.evaluate()
print("\nEvaluation Results:")
for key, value in results.items():
    print(key, ":", value)
print("\n")
print("============================================================")
print("FINAL PERFORMANCE")
print("============================================================")

print(
    "Accuracy:",
    round(results["eval_accuracy"] * 100, 2),
    "%"
)
print(
    "Precision:",
    round(results["eval_precision"] * 100, 2),
    "%"
)
print(
    "Recall:",
    round(results["eval_recall"] * 100, 2),
    "%"
)
print(
    "F1 Score:",
    round(results["eval_f1"] * 100, 2),
    "%"
)
print("\n")
print("============================================================")
print("GENERATING TEST PREDICTIONS")
print("============================================================")

prediction_output = trainer.predict(tokenized_test)

predictions = np.argmax(
    prediction_output.predictions,
    axis=1
)

actual_labels = test_df["label"].values

print("\n")
print("============================================================")
print("CLASSIFICATION REPORT")
print("============================================================")

print(
    classification_report(
        actual_labels,
        predictions,
        labels=[0, 1, 2, 3],
        target_names=[
            "World",
            "Sports",
            "Business",
            "Sci/Tech"
        ],
        zero_division=0
    )
)
print("\n")
print("============================================================")
print("CONFUSION MATRIX")
print("============================================================")

cm = confusion_matrix(
    actual_labels,
    predictions,
    labels=[0, 1, 2, 3]
)

print(cm)
print("\n")
print("============================================================")
print("SAVING MODEL")
print("============================================================")

model_path = "./news_classifier_model"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print("Model saved successfully!")
print("Location:", model_path)

def predict_news(news_text):
    inputs = tokenizer(
        news_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )
    prediction = torch.argmax(
        probabilities,
        dim=1
    ).item()
    confidence = probabilities[0, prediction].item()
    category = id2label[prediction]
    return category, confidence

print("\n")
print("============================================================")
print("SAMPLE NEWS PREDICTIONS")
print("============================================================")


news1 = """
The stock market rose sharply today after major companies
reported strong quarterly profits.
"""

category, confidence = predict_news(news1)

print("\nNews 1:", news1)
print("Predicted Category:", category)
print("Confidence:", round(confidence * 100, 2), "%")


news2 = """
The football team won the championship after defeating
its opponent in the final match.
"""
category, confidence = predict_news(news2)

print("\nNews 2:", news2)
print("Predicted Category:", category)
print("Confidence:", round(confidence * 100, 2), "%")

news3 = """
Scientists developed a new computer technology that can
improve artificial intelligence systems.
"""
category, confidence = predict_news(news3)

print("\nNews 3:", news3)
print("Predicted Category:", category)
print("Confidence:", round(confidence * 100, 2), "%")

news4 = """
World leaders met to discuss international relations
and global security.
"""
category, confidence = predict_news(news4)

print("\nNews 4:", news4)
print("Predicted Category:", category)
print("Confidence:", round(confidence * 100, 2), "%")

print("\n")
print("============================================================")
print("CUSTOM NEWS CLASSIFIER")
print("============================================================")

print("Enter your own news article.")
print("Type 'exit' to stop.")

while True:
    user_news = input("\nEnter news text: ")
    if user_news.lower().strip() == "exit":
        print("\nProgram completed.")
        break
    if user_news.strip() == "":
        print("Please enter some news text.")
        continue
    category, confidence = predict_news(user_news)
    print("\nPredicted Category:", category)
    print(
        "Confidence:",
        round(confidence * 100, 2),
        "%"
    )


Libraries imported successfully!

================ DEVICE INFORMATION ================
Using CPU

================ DATASET INFORMATION ================
Train Shape: (120000, 3)
Test Shape: (7600, 3)

Train Columns:
['Class Index', 'Title', 'Description']

Test Columns:
['Class Index', 'Title', 'Description']

First 5 Training Records:


,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."



Columns after renaming:
['label', 'title', 'description']

================ MISSING VALUES ================

Train Missing Values:
label          0
title          0
description    0
dtype: int64

Test Missing Values:
label          0
title          0
description    0
dtype: int64

================ LABEL DISTRIBUTION ================
label
1    30000
2    30000
3    30000
4    30000
Name: count, dtype: int64

================ REDUCING DATASET ================
Reduced Training Shape: (8000, 4)
Reduced Testing Shape: (2000, 4)

================ CLEAN DATA ================


,text,label
0,"BBC set for major shake-up, claims newspaper L...",2
1,Marsh averts cash crunch Embattled insurance b...,2
2,"Jeter, Yankees Look to Take Control (AP) AP - ...",1
3,Flying the Sun to Safety When the Genesis caps...,3
4,Stocks Seen Flat as Nortel and Oil Weigh NEW ...,2


Train Shape: (8000, 2)
Test Shape: (2000, 2)

================ HUGGING FACE DATASET ================
Training Dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 8000
})
Testing Dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 2000
})

================ TOKENIZER ================


Tokenizer loaded successfully!

Tokenizing training data...


... (3 outputs left out)

Testing tokenization completed!
Tokenized datasets ready!

================ MODEL ================


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2120.61it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded successfully!

Training arguments created successfully!
Trainer created successfully!


STARTING DISTILBERT TRAINING


Epoch,Training Loss,Validation Loss
